# 03 · Prepare Hazards Dataset (zebra crossings, signs, waterlogging)

> **OWNER:** Member B (M4 · hazards + plates)
> **PREREQUISITES:** `00_setup_verify.ipynb` all-green. **START THIS NOTEBOOK FIRST**
> — it is the slowest human task on the whole ML track and it does not parallelise
> with a GPU, so get the sourcing/labelling clock running before anything else.
> **EXPECTED RUNTIME:** 3-4 hours, and it is mostly you, not the machine —
> sourcing images and drawing boxes in Roboflow.
> **OUTPUTS:** `data/hazards/{images,labels}/{train,val,test}/` + `data.yaml`,
> a contact sheet, and a per-class count table.

No public Indian dataset covers faded zebra crossings, damaged signs, and
waterlogging together — this notebook is where that dataset gets built,
merging class-specific sources. `DAMAGED_DIVIDER` was dropped (2026-08-28):
the one Indian-specific lead (DATS_2022) could not be verified — its file API
is inaccessible without a browser session, and its own paper never lists
"divider" as one of its 45 annotated classes, only as background-scene prose.
`WATERLOGGING` stays in despite foreign-geography source data — see the
explicit caveat in Step 1 and the eventual model card.

**Next notebook:** `04_train_hazards.ipynb` (yours too — M4).

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Sourcing guidance (read this, do not scrape automatically)

Target **~100 images per class, ~300 total** across the three classes below.
`DAMAGED_DIVIDER` was dropped (2026-08-28) — its one Indian-specific lead
(DATS_2022, Mendeley) could not be verified as actually having a labelled
divider class, and no other candidate had a confirmable licence.

**`WATERLOGGING` stays in with an explicit caveat: the sourced imagery is
foreign-road (US), not Indian.** If asked in Q&A: "trained on non-Indian
flood imagery, domain gap acknowledged" — say that rather than let it be
discovered. This caveat must also land in `MODEL_CARD_hazard.md` (notebook 04).

Sourcing options, roughly cheapest-first:

1. **Roboflow Universe, Kaggle, Hugging Face, Mendeley Data, GitHub** — search
   each class name (`zebra crossing` / `pedestrian crossing`, `damaged
   traffic sign` / `vandalized sign`, `waterlogging` / `flooded road`).
   Datasets for each come from different sources — that's fine, merge them.
   **Check each dataset's licence before use** — licences range from CC BY
   4.0 to CC BY-NC-ND/CC BY-NC-SA, and this project needs redistribution/
   training rights, not research-only.
2. **Your own dashcam footage** — drop raw video into `data/raw_video/`
   (already gitignored) and use the frame-extraction helper in Step 2 below,
   if/when footage becomes available.

Note on `DAMAGED_SIGN`: this class covers **visibly damaged/bent/faded/
vandalized signs**, not a "sign is missing" absence class — a bounding-box
detector cannot box something that isn't there. True missing-sign inference
belongs in a future expected-vs-observed fusion layer (comparing OpenStreetMap
infrastructure against what buses observe on a segment), not this notebook.

This step is guidance, not automation — go source images now, then come back
for Step 2 (frame extraction, if you have video) or skip straight to Step 3
(Roboflow) once you have images to annotate.

In [ ]:
TARGET_PER_CLASS = 100
TARGET_TOTAL = 300
MINIMUM_PER_CLASS = 50  # Step 6 refuses to proceed below this, per class

print(f"target: ~{TARGET_PER_CLASS} images/class, ~{TARGET_TOTAL} total (3 classes)")
print(f"hard minimum before this notebook lets you proceed: {MINIMUM_PER_CLASS}/class")
print("if time runs short: WATERLOGGING is the most defensible to drop (foreign-geography")
print("source data, see Step 1) — keep FADED_ZEBRA and DAMAGED_SIGN well-annotated first")

## Step 2 — Frame extraction (only if you have dashcam video)

Samples every Nth frame, drops near-identical frames via a perceptual
(average) hash so you are not annotating forty near-duplicates of one
waterlogged junction, and exports to a labelling folder ready for Roboflow upload.

In [ ]:
import cv2
from PIL import Image


def _average_hash(image: Image.Image, hash_size: int = 8) -> int:
    small = image.convert("L").resize((hash_size, hash_size), Image.LANCZOS)
    pixels = list(small.getdata())
    avg = sum(pixels) / len(pixels)
    bits = "".join("1" if p > avg else "0" for p in pixels)
    return int(bits, 2)


def _hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


def extract_and_dedupe_frames(video_path, output_dir, every_n=15, hash_threshold=5, max_frames=None):
    """Samples every `every_n`th frame, keeps it only if its perceptual hash
    differs from every frame already kept by more than `hash_threshold` bits."""
    video_path, output_dir = Path(video_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"could not open {video_path}")

    seen_hashes, saved, frame_idx = [], 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % every_n == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb)
            h = _average_hash(img)
            if all(_hamming(h, seen) > hash_threshold for seen in seen_hashes):
                seen_hashes.append(h)
                img.save(output_dir / f"{video_path.stem}_{frame_idx:06d}.jpg", quality=95)
                saved += 1
                if max_frames and saved >= max_frames:
                    break
        frame_idx += 1
    cap.release()
    print(f"{video_path.name}: scanned {frame_idx} frames, sampled every {every_n}, kept {saved} after dedupe (hash threshold={hash_threshold})")
    return saved


RAW_VIDEO_DIR = REPO_ROOT / "data" / "raw_video"
LABELLING_DIR = DATA_ROOT / "hazards" / "to_label"
LABELLING_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(RAW_VIDEO_DIR.glob("*.mp4")) + sorted(RAW_VIDEO_DIR.glob("*.mov"))
if not videos:
    print(f"no video found in {RAW_VIDEO_DIR} — drop dashcam footage there and re-run, or skip to Step 3 (Roboflow) if you are only using sourced images.")
else:
    for v in videos:
        extract_and_dedupe_frames(v, LABELLING_DIR, every_n=15, hash_threshold=5)
    print(f"frames ready for upload/annotation in {LABELLING_DIR}")

## Step 3 — Roboflow annotation workflow (manual)

1. Create a Roboflow project (Object Detection).
2. Upload sourced images (from Step 1's Universe datasets, or `LABELLING_DIR`
   from Step 2 if you extracted your own frames).
3. Annotate **bounding boxes only — do not use segmentation.** Masks are
   5-10x slower to annotate and nothing downstream (the `Observation.bbox`
   contract, the ByteTrack-based trackers, the fusion pipeline) consumes
   pixel-level extent — a box is all the system ever asks for.
4. Class names in Roboflow **must exactly match** the frozen names below —
   name them identically or the import step in Step 4 will report them as unknown.
5. Generate a version. Augmentation is fine to apply in Roboflow (it does not
   affect the frozen indices), but keep your **raw, unaugmented** annotations
   too if you want to redo the split differently later.
6. Export -> **YOLOv11** format (a plain YOLO txt export; Roboflow's "YOLOv11"
   and "YOLOv8" export presets are identical box-format, this just needs any
   YOLO-txt-format export).
7. Download the export zip into `data/hazards/roboflow_export.zip` (or
   extracted directly into `data/hazards/roboflow_export/`).

In [ ]:
from common import constants

print("Roboflow class names must exactly match:")
for idx, name in constants.HAZARD_CLASSES.items():
    print(f"  {idx}: {name}")

## Step 4 — Import the Roboflow export, verify frozen indices, stratified split

In [ ]:
import shutil
import zipfile

HAZARDS_ROOT = DATA_ROOT / "hazards"
export_zip = HAZARDS_ROOT / "roboflow_export.zip"
export_dir = HAZARDS_ROOT / "roboflow_export"

if export_zip.exists() and not export_dir.exists():
    with zipfile.ZipFile(export_zip) as zf:
        zf.extractall(export_dir)
    print(f"extracted {export_zip} -> {export_dir}")

if not export_dir.exists():
    raise FileNotFoundError(
        f"no Roboflow export found at {export_zip} or {export_dir}. "
        "Complete Step 3 (Roboflow annotation + export) first."
    )

import yaml

roboflow_yaml_path = next(export_dir.rglob("data.yaml"), None)
if roboflow_yaml_path is None:
    raise FileNotFoundError(f"no data.yaml found under {export_dir} — check the export extracted correctly")

with open(roboflow_yaml_path) as f:
    roboflow_yaml = yaml.safe_load(f)

roboflow_names = roboflow_yaml["names"]
roboflow_index_by_name = {name: idx for idx, name in (roboflow_names.items() if isinstance(roboflow_names, dict) else enumerate(roboflow_names))}
print(f"Roboflow export classes: {roboflow_index_by_name}")

REMAP_NEEDED = roboflow_index_by_name != constants.HAZARD_CLASSES
if REMAP_NEEDED:
    print()
    print("Roboflow's class order does not match the frozen indices — remapping label files (not reordering constants.py).")
    missing = set(constants.HAZARD_CLASSES.values()) - set(roboflow_index_by_name)
    if missing:
        raise AssertionError(f"Roboflow export is missing frozen class(es): {missing} — check spelling matched Step 3 exactly.")

In [ ]:
# Remap every label file from Roboflow's index order to the frozen index order (a no-op if they already match).
roboflow_to_frozen = {roboflow_index_by_name[name]: idx for idx, name in constants.HAZARD_CLASSES.items()}

roboflow_images_dir = next(export_dir.rglob("images"), None) or next((p.parent / "images" for p in export_dir.rglob("*.txt")), None)
roboflow_labels_dir = next(export_dir.rglob("labels"), None)
if roboflow_images_dir is None or roboflow_labels_dir is None:
    raise FileNotFoundError(f"could not locate images/ and labels/ under {export_dir} — inspect the export layout by hand")

REMAPPED_LABELS_DIR = HAZARDS_ROOT / "labels_remapped"
REMAPPED_LABELS_DIR.mkdir(parents=True, exist_ok=True)

image_class_map = {}
for label_path in roboflow_labels_dir.rglob("*.txt"):
    lines_out = []
    classes = set()
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        old_idx = int(parts[0])
        new_idx = roboflow_to_frozen.get(old_idx)
        if new_idx is None:
            continue  # class not in the frozen set — dropped, not silently kept
        classes.add(new_idx)
        lines_out.append(" ".join([str(new_idx), *parts[1:]]))
    (REMAPPED_LABELS_DIR / label_path.name).write_text("\n".join(lines_out) + ("\n" if lines_out else ""))
    image_class_map[label_path.stem] = classes

print(f"remapped {len(image_class_map)} label files -> {REMAPPED_LABELS_DIR}")

In [ ]:
from common import splits

data_splits = splits.stratified_split(image_class_map, ratios=(0.8, 0.1, 0.1), seed=42)
splits.report_split_balance(image_class_map, data_splits, constants.HAZARD_CLASSES)

In [ ]:
OUTPUT_ROOT = DATA_ROOT / "hazards_prepared"
splits.materialize_split(
    image_class_map=image_class_map,
    splits=data_splits,
    images_src=roboflow_images_dir,
    labels_src=REMAPPED_LABELS_DIR,
    output_root=OUTPUT_ROOT,
)

data_yaml = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": constants.HAZARD_CLASSES,
}
constants.assert_class_order(data_yaml["names"], constants.HAZARD_CLASSES, "hazard")

data_yaml_path = OUTPUT_ROOT / "data.yaml"
data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"wrote {data_yaml_path}")

## Step 5 — Contact sheet — LOOK AT THIS BEFORE ANYTHING ELSE

Same warning as notebook 01: offset or inverted boxes mean the Roboflow
export/remap step is broken. Ten seconds here saves a day of training on garbage.

In [ ]:
from common import contact_sheet

sheet_path = contact_sheet.render_contact_sheet(
    images_dir=OUTPUT_ROOT / "images" / "train",
    labels_dir=OUTPUT_ROOT / "labels" / "train",
    class_names=constants.HAZARD_CLASSES,
    output_path=OUTPUT_ROOT / "contact_sheet.png",
    n=12,
)

from IPython.display import Image as IPImage, display

display(IPImage(filename=str(sheet_path)))

## Step 6 — Per-class count table. REFUSES to proceed under 50 images/class.

An accuracy number reported off a class with a handful of images is not a
real number — this gate exists so that gets caught here, not in the deck.

In [ ]:
from collections import Counter

per_class_images = Counter()
for classes in image_class_map.values():
    for c in classes:
        per_class_images[c] += 1

print(f"{'class':<20}{'images':>10}")
under_minimum = []
for idx, name in constants.HAZARD_CLASSES.items():
    n = per_class_images.get(idx, 0)
    print(f"{name:<20}{n:>10}")
    if n < MINIMUM_PER_CLASS:
        under_minimum.append((name, n))

if under_minimum:
    raise AssertionError(
        f"Under the {MINIMUM_PER_CLASS}-image minimum for: {under_minimum}. "
        "A per-class accuracy number below this is not a reportable claim — go back to "
        "Step 1/3 and source more for these classes (or drop WATERLOGGING if it is the "
        "one short, per the guidance in Step 1) before training on this data."
    )
print()
print("all classes clear the minimum — proceed to 04_train_hazards.ipynb")

---
### What this notebook produced
- `data/hazards_prepared/{images,labels}/{train,val,test}/` + `data.yaml` (gitignored)
- A contact sheet you looked at and confirmed looks correct
- A per-class count table that passed the 50-image minimum

### Next
`04_train_hazards.ipynb` (yours too — M4).